In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install numpy pandas tensorflow scikit-learn scikit-survival

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 125.0 MB/s eta 0:00:00
  Created wheel for ecos: filename=ecos-2.0.14-cp313-cp313-linux_x86_64.whl size=204008 sha256=02be10e6aaee65f62b15b36c6996f4eee97fc943aafb9caa14ea152ee2bf26f1
  Stored in directory: /root/.cache/pip/wheels/6b/82/0b/4bb5aa6c4618f367601a45db8710205d56858808423974e92a
Successfully built ecos
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [3]:
# ============================================================
# REAL DATASETS ONLY: JOE + KM SURVIVAL EXPERIMENTS
#
# Methods:
#   1. NN OHE
#   2. KM Greedy
#   3. JOEOH 3SIG
#   4. JOEOH 1LIN
#   5. JOECHAR 3SIG
#   6. JOECHAR 1LIN
#
# CIBMTR:
#   categorical variables are kept separate (cat_1, cat_2, ...)
#   using data_dictionary.csv; NO joint category is created.
#
# Metrics:
#   - C-index
#   - Integrated Brier Score (IBS)
#
# Requires:
#   pip install numpy pandas tensorflow scikit-learn scikit-survival
#
# Also requires:
#   R + your existing rcode_full_km.R script
#
# ============================================================

from __future__ import annotations

import os
import gc
import shutil
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sksurv.metrics import (
    concordance_index_censored,
    integrated_brier_score,
)

from sksurv.linear_model.coxph import BreslowEstimator


# ============================================================
# 1. USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# IMPORTANT:
# Change this to the location of your R KM encoder.
# ------------------------------------------------------------

R_ENCODER_SCRIPT = (
    "/content/drive/MyDrive/Colab Notebooks/rcode_full_km_multi_cat_ready.R"
)


# ------------------------------------------------------------
# Folder containing the real datasets.
#
# If your CSV files are in the current Colab working directory,
# simply use:
#
# DATA_DIR = "."
#
# ------------------------------------------------------------

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/"


# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

RESULTS_ROOT = "results_real_joe_km"


# ------------------------------------------------------------
# Experiment settings
# ------------------------------------------------------------

N_REPEATS = 20

TRAIN_FRAC = 0.5

ENCODING_DIM = 1

EPOCHS = 500

LR = 0.05




# ------------------------------------------------------------
# FALSE reproduces your current R random splitting more closely.
#
# TRUE keeps approximately the same event fraction in train/test.
# ------------------------------------------------------------

STRATIFY_EVENT = False


# ------------------------------------------------------------
# Choose datasets.
#
# Use one:
#
# DATASETS_TO_RUN = ["TELCO"]
#
# Or several:
#
# DATASETS_TO_RUN = ["METABRIC", "SUPPORT"]
#
# Or all:
# ------------------------------------------------------------

DATASETS_TO_RUN = [
    "SUPPORT"
]


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_seed(seed: int) -> None:

    np.random.seed(seed)

    tf.random.set_seed(seed)


# ============================================================
# 3. DATASET CONFIGURATION
# ============================================================

@dataclass
class DatasetConfig:

    name: str

    file: str

    time_col: str

    event_col: str

    continuous_cols: List[str]

    categorical_cols: List[str]

    id_cols: Optional[List[str]] = None

    positive_event_labels: Optional[List[str]] = None


DATASETS: Dict[str, DatasetConfig] = {

    # ========================================================
    # METABRIC
    # ========================================================

    "METABRIC": DatasetConfig(

        name="METABRIC",

        file="df_dataset.csv",

        time_col="duration",

        event_col="event",

        continuous_cols=[
            "x0",
            "x1",
            "x2",
            "x3",
            "x8",
        ],

        categorical_cols=[
            "x4",
            "x5",
            "x6",
            "x7",
        ],

        id_cols=None,

        positive_event_labels=[
            "1",
        ],
    ),


    # ========================================================
    # SUPPORT
    # ========================================================

    "SUPPORT": DatasetConfig(

        name="SUPPORT",

        file="df_dataset_support.csv",

        time_col="duration",

        event_col="event",

        continuous_cols=[
            "x0",
            "x7",
            "x8",
            "x9",
            "x10",
            "x11",
            "x12",
            "x13",
        ],

        categorical_cols=[
            "x1",
            "x2",
            "x3",
            "x4",
            "x5",
            "x6",
        ],

        id_cols=None,

        positive_event_labels=[
            "1",
        ],
    ),


    # ========================================================
    # TELCO
    #
    # Matches your R code after dropping:
    #
    # MultipleLines
    # OnlineSecurity
    # OnlineBackup
    # DeviceProtection
    # TechSupport
    # StreamingTV
    # StreamingMovies
    # ========================================================

    "TELCO": DatasetConfig(

        name="TELCO",

        file="customer_data (1).csv",

        time_col="tenure",

        event_col="Churn",

        continuous_cols=[
            "MonthlyCharges",
            "TotalCharges",
        ],

        categorical_cols=[
            "gender",
            "SeniorCitizen",
            "Partner",
            "Dependents",
            "PhoneService",
            "InternetService",
            "Contract",
            "PaperlessBilling",
            "PaymentMethod",
        ],

        id_cols=[
            "customerID",
        ],

        positive_event_labels=[
            "YES",
        ],
    ),


    # ========================================================
    # GBSG
    # ========================================================

    "GBSG": DatasetConfig(

        name="GBSG",

        file="df_dataset_gbsg.csv",

        time_col="duration",

        event_col="event",

        continuous_cols=[
            "x3",
            "x4",
            "x5",
            "x6",
        ],

        categorical_cols=[
            "x0",
            "x1",
            "x2",
        ],

        id_cols=None,

        positive_event_labels=[
            "1",
        ],
    ),


    # ========================================================
    # BREAST CANCER
    # ========================================================

    "BREAST_CANCER": DatasetConfig(

        name="BREAST_CANCER",

        file="Breast_Cancer.csv",

        time_col="Survival Months",

        event_col="Status",

        continuous_cols=[
            "Age",
            "Tumor Size",
            "Regional Node Examined",
            "Reginol Node Positive",
        ],

        categorical_cols=[
            "Race",
            "Marital Status",
            "T Stage ",
            "N Stage",
            "6th Stage",
            "differentiate",
            "Grade",
            "A Stage",
            "Estrogen Status",
            "Progesterone Status",
        ],

        id_cols=None,

        positive_event_labels=[
            "DEAD",
        ],
    ),

    # ========================================================
    # CIBMTR
    #
    # Predictor types are read dynamically from
    # data_dictionary.csv. Categorical predictors are NOT
    # collapsed into a joint category.
    # ========================================================

    "CIBMTR": DatasetConfig(

        name="CIBMTR",

        file="CIBMTR.csv",

        time_col="efs_time",

        event_col="efs",

        # Filled dynamically from data_dictionary.csv.
        continuous_cols=[],

        categorical_cols=[],

        id_cols=[
            "ID",
        ],

        positive_event_labels=[
            "1",
        ],
    ),

}


# ============================================================
# 4. EVENT ENCODING
# ============================================================

def encode_event(
    series: pd.Series,
    positive_labels: Optional[List[str]],
) -> pd.Series:

    """
    Convert the event column to numeric 0/1.

    Numeric input:
        1 -> event
        anything else -> censored

    Character input:
        values in positive_event_labels -> event
        everything else -> censored
    """

    # --------------------------------------------------------
    # Numeric event column
    # --------------------------------------------------------

    if pd.api.types.is_numeric_dtype(series):

        numeric = pd.to_numeric(
            series,
            errors="coerce",
        )

        result = pd.Series(
            np.nan,
            index=series.index,
            dtype=float,
        )

        valid = numeric.notna()

        result.loc[valid] = (
            numeric.loc[valid] == 1
        ).astype(float)

        return result


    # --------------------------------------------------------
    # Character event column
    # --------------------------------------------------------

    positives = {

        str(value)
        .strip()
        .upper()

        for value in (
            positive_labels or ["1"]
        )
    }


    result = pd.Series(
        np.nan,
        index=series.index,
        dtype=float,
    )


    missing = series.isna()

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.upper()
    )


    result.loc[~missing] = (
        cleaned.loc[~missing]
        .isin(positives)
        .astype(float)
    )

    return result


# ============================================================
# 5. REAL DATASET PREPROCESSING
# ============================================================

def load_real_dataset(
    config: DatasetConfig,
) -> Tuple[
    pd.DataFrame,
    List[str],
    List[str],
]:

    """
    Convert a real dataset to the common experiment representation:

        cont_1
        cont_2
        ...
        cat_1
        Time
        Event

    All original categorical variables are combined into ONE
    joint categorical variable, matching your R experiment.
    """

    file_path = os.path.join(
        DATA_DIR,
        config.file,
    )


    if not os.path.exists(file_path):

        raise FileNotFoundError(

            f"\nDataset file not found:\n"
            f"{file_path}\n\n"
            f"Check DATA_DIR and the filename."
        )


    print("\n")
    print("=" * 75)
    print(f"LOADING DATASET: {config.name}")
    print("=" * 75)

    print(
        f"File: {file_path}"
    )


    df = pd.read_csv(
        file_path,
        low_memory=False,
    )


    print(
        f"Raw dimensions: "
        f"{df.shape[0]} x {df.shape[1]}"
    )


    # ========================================================
    # Remove ID columns
    # ========================================================

    if config.id_cols is not None:

        drop_cols = [

            col

            for col
            in config.id_cols

            if col in df.columns
        ]

        if len(drop_cols) > 0:

            df = df.drop(
                columns=drop_cols
            )


    # ========================================================
    # Check required columns
    # ========================================================

    required_columns = (

        config.continuous_cols

        + config.categorical_cols

        + [
            config.time_col,
            config.event_col,
        ]
    )


    missing_columns = [

        col

        for col in required_columns

        if col not in df.columns
    ]


    if len(missing_columns) > 0:

        raise ValueError(

            f"\n{config.name} is missing required columns:\n"
            f"{missing_columns}\n\n"
            f"Available columns:\n"
            f"{list(df.columns)}"
        )


    # ========================================================
    # Convert continuous variables to numeric
    # ========================================================

    for col in config.continuous_cols:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce",
        )


    # ========================================================
    # Convert Time
    # ========================================================

    df[config.time_col] = pd.to_numeric(
        df[config.time_col],
        errors="coerce",
    )


    # ========================================================
    # Convert Event
    # ========================================================

    df[config.event_col] = encode_event(

        df[config.event_col],

        config.positive_event_labels,
    )


    # ========================================================
    # Remove missing continuous predictors
    #
    # Matches the approach in your R real-data code.
    # ========================================================

    n_before = len(df)


    df = df.dropna(
        subset=config.continuous_cols
    ).copy()


    print(
        "Rows removed for missing continuous variables:",
        n_before - len(df),
    )


    # ========================================================
    # Remove missing Time/Event
    # ========================================================

    n_before = len(df)


    df = df.dropna(
        subset=[
            config.time_col,
            config.event_col,
        ]
    ).copy()


    print(
        "Rows removed for missing Time/Event:",
        n_before - len(df),
    )


    # ========================================================
    # Require non-negative survival time
    # ========================================================

    df = df[
        df[config.time_col] >= 0
    ].copy()


    df[config.event_col] = (
        df[config.event_col]
        .astype(int)
    )


    # ========================================================
    # Missing categorical values become "NA"
    # ========================================================

    for col in config.categorical_cols:

        values = df[col].copy()

        values = values.where(
            values.notna(),
            "NA",
        )

        values = (
            values
            .astype(str)
            .str.strip()
        )

        values = values.replace(
            "",
            "NA",
        )

        df[col] = values


    # ========================================================
    # Build joint categorical variable
    #
    # Equivalent to:
    #
    # paste(row, collapse = " | ")
    #
    # in your R implementation.
    # ========================================================

    df["cat_1"] = (

        df[
            config.categorical_cols
        ]

        .astype(str)

        .agg(
            " | ".join,
            axis=1,
        )
    )


    # ========================================================
    # Rename continuous variables
    # ========================================================

    continuous_rename = {

        original:
            f"cont_{i + 1}"

        for i, original
        in enumerate(
            config.continuous_cols
        )
    }


    df = df.rename(
        columns=continuous_rename
    )


    cont_cols = list(
        continuous_rename.values()
    )


    # ========================================================
    # Rename survival variables
    # ========================================================

    df = df.rename(

        columns={

            config.time_col:
                "Time",

            config.event_col:
                "Event",
        }
    )


    # ========================================================
    # Keep only experiment columns
    # ========================================================

    keep_columns = (

        cont_cols

        + [
            "cat_1",
            "Time",
            "Event",
        ]
    )


    df = (

        df[
            keep_columns
        ]

        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # Diagnostics
    # ========================================================

    print("\nProcessed dataset")

    print(
        "Rows:",
        len(df),
    )

    print(
        "Continuous variables:",
        len(cont_cols),
    )

    print(
        "Joint categorical levels:",
        df["cat_1"].nunique(),
    )

    print(
        "Events:",
        int(
            df["Event"].sum()
        ),
    )

    print(
        "Censored:",
        int(
            (df["Event"] == 0).sum()
        ),
    )

    print(
        "Event fraction:",
        round(
            float(
                df["Event"].mean()
            ),
            4,
        ),
    )

    print(
        "Minimum Time:",
        df["Time"].min(),
    )

    print(
        "Maximum Time:",
        df["Time"].max(),
    )


    if len(cont_cols) == 0:

        raise ValueError(
            f"{config.name}: "
            "no continuous variables available."
        )


    if df["cat_1"].nunique() < 2:

        raise ValueError(
            f"{config.name}: "
            "fewer than two categorical levels."
        )


    if df["Event"].sum() == 0:

        raise ValueError(
            f"{config.name}: "
            "there are no events."
        )


    cat_cols = ["cat_1"]

    return (
        df,
        cont_cols,
        cat_cols,
    )


# ============================================================
# 5B. CIBMTR PREPROCESSING — KEEP CATEGORIES SEPARATE
# ============================================================

def load_cibmtr_dataset(
    cibmtr_file: str,
    dictionary_file: str,
) -> Tuple[
    pd.DataFrame,
    List[str],
    List[str],
]:
    """
    Load CIBMTR using data_dictionary.csv.

    Unlike the other real datasets, CIBMTR keeps each categorical
    predictor separate:

        cat_1, cat_2, ..., cat_p

    No joint categorical feature is created.
    """

    if not os.path.exists(cibmtr_file):
        raise FileNotFoundError(
            f"\nCIBMTR file not found:\n{cibmtr_file}"
        )

    if not os.path.exists(dictionary_file):
        raise FileNotFoundError(
            f"\nCIBMTR dictionary not found:\n{dictionary_file}"
        )

    print("\n")
    print("=" * 75)
    print("LOADING DATASET: CIBMTR")
    print("=" * 75)
    print(f"File: {cibmtr_file}")
    print(f"Dictionary: {dictionary_file}")

    df = pd.read_csv(
        cibmtr_file,
        low_memory=False,
    )

    dictionary = pd.read_csv(
        dictionary_file,
        low_memory=False,
    )

    print(
        f"Raw dimensions: "
        f"{df.shape[0]} x {df.shape[1]}"
    )

    # --------------------------------------------------------
    # Clean dictionary
    # --------------------------------------------------------

    dictionary.columns = [
        str(col).strip()
        for col in dictionary.columns
    ]

    required_dictionary_cols = {
        "variable",
        "type",
    }

    if not required_dictionary_cols.issubset(
        dictionary.columns
    ):
        raise ValueError(
            "data_dictionary.csv must contain columns "
            "'variable' and 'type'."
        )

    dictionary["variable"] = (
        dictionary["variable"]
        .astype(str)
        .str.strip()
    )

    dictionary["type"] = (
        dictionary["type"]
        .astype(str)
        .str.strip()
    )

    # Retain only dictionary variables present in CIBMTR.csv.
    dictionary = dictionary[
        dictionary["variable"].isin(df.columns)
    ].copy()

    id_col = "ID"
    time_col = "efs_time"
    event_col = "efs"

    # --------------------------------------------------------
    # Determine predictor types from dictionary
    # --------------------------------------------------------

    original_cat_cols = dictionary.loc[
        dictionary["type"].str.casefold() == "categorical",
        "variable",
    ].tolist()

    original_cont_cols = dictionary.loc[
        dictionary["type"].str.casefold() == "numerical",
        "variable",
    ].tolist()

    excluded = {
        id_col,
        time_col,
        event_col,
    }

    original_cat_cols = [
        col
        for col in original_cat_cols
        if col not in excluded
    ]

    original_cont_cols = [
        col
        for col in original_cont_cols
        if col not in excluded
    ]

    if len(original_cat_cols) == 0:
        raise ValueError(
            "No CIBMTR categorical predictors were identified "
            "from data_dictionary.csv."
        )

    if len(original_cont_cols) == 0:
        raise ValueError(
            "No CIBMTR numerical predictors were identified "
            "from data_dictionary.csv."
        )

    # --------------------------------------------------------
    # Continuous predictors
    # --------------------------------------------------------

    for col in original_cont_cols:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce",
        )

    # --------------------------------------------------------
    # Outcome
    # --------------------------------------------------------

    df[time_col] = pd.to_numeric(
        df[time_col],
        errors="coerce",
    )

    df[event_col] = pd.to_numeric(
        df[event_col],
        errors="coerce",
    )

    # --------------------------------------------------------
    # Missing categorical values -> explicit "NA"
    # --------------------------------------------------------

    for col in original_cat_cols:

        values = df[col].copy()

        values = values.where(
            values.notna(),
            "NA",
        )

        values = (
            values
            .astype(str)
            .str.strip()
        )

        values = values.replace(
            "",
            "NA",
        )

        df[col] = values

    # --------------------------------------------------------
    # Match the other real-data preprocessing:
    # remove rows missing numerical predictors or outcome.
    # --------------------------------------------------------

    n_before = len(df)

    df = df.dropna(
        subset=original_cont_cols
    ).copy()

    print(
        "Rows removed for missing continuous variables:",
        n_before - len(df),
    )

    n_before = len(df)

    df = df.dropna(
        subset=[
            time_col,
            event_col,
        ]
    ).copy()

    print(
        "Rows removed for missing Time/Event:",
        n_before - len(df),
    )

    df = df[
        df[time_col] >= 0
    ].copy()

    df[event_col] = (
        df[event_col] == 1
    ).astype(int)

    # --------------------------------------------------------
    # Rename each categorical predictor separately
    # --------------------------------------------------------

    cat_rename = {
        original: f"cat_{i + 1}"
        for i, original
        in enumerate(original_cat_cols)
    }

    cont_rename = {
        original: f"cont_{i + 1}"
        for i, original
        in enumerate(original_cont_cols)
    }

    df = df.rename(
        columns={
            **cat_rename,
            **cont_rename,
            time_col: "Time",
            event_col: "Event",
        }
    )

    cat_cols = list(
        cat_rename.values()
    )

    cont_cols = list(
        cont_rename.values()
    )

    keep_cols = (
        cont_cols
        + cat_cols
        + [
            "Time",
            "Event",
        ]
    )

    df = (
        df[keep_cols]
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Diagnostics
    # --------------------------------------------------------

    print("\nProcessed CIBMTR dataset")
    print("Rows:", len(df))
    print("Continuous variables:", len(cont_cols))
    print("Categorical variables:", len(cat_cols))
    print("Events:", int(df["Event"].sum()))
    print(
        "Event fraction:",
        round(
            float(df["Event"].mean()),
            4,
        ),
    )

    print("\nCategorical cardinalities:")

    for original, renamed in cat_rename.items():
        print(
            f"  {renamed} ({original}): "
            f"{df[renamed].nunique()} levels"
        )

    if df["Event"].sum() == 0:
        raise ValueError(
            "CIBMTR contains no events after preprocessing."
        )

    return (
        df,
        cont_cols,
        cat_cols,
    )


# ============================================================
# 6. ONE-HOT ENCODING
# ============================================================

def make_one_hot_encoder():

    """
    Compatibility helper for different sklearn versions.
    """

    try:

        encoder = OneHotEncoder(

            handle_unknown="ignore",

            drop="first",

            sparse_output=False,
        )

    except TypeError:

        # Older sklearn
        encoder = OneHotEncoder(

            handle_unknown="ignore",

            drop="first",

            sparse=False,
        )

    return encoder


def build_ohe_arrays_r_style(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cat_cols: List[str],
) -> Tuple[
    np.ndarray,
    np.ndarray,
    Dict,
]:
    """
    Reproduce the R OHE logic:

    - Levels determined from training data only.
    - A level is valid only if at least one Event == 1
      occurs for that level in the training data.
    - First valid level = baseline.
    - Remaining valid levels get dummy columns.
    - Baseline, fully censored levels, and unseen test levels
      all map to all-zero.
    - If a categorical variable has fewer than 2 valid levels,
      it contributes no OHE columns.
    """

    train_blocks = []
    test_blocks = []

    metadata = {}

    for col in cat_cols:

        train_cats = train_df[col].astype(str).to_numpy()
        test_cats = test_df[col].astype(str).to_numpy()

        # ------------------------------------------------------
        # Match R:
        # all_levels <- unique(train_cats)
        #
        # pd.unique preserves order of appearance,
        # like R's unique().
        # ------------------------------------------------------

        all_levels = list(pd.unique(train_cats))

        # ------------------------------------------------------
        # Check whether each level has at least one event
        # ------------------------------------------------------

        valid_levels = []
        censored_levels = []

        for level in all_levels:

            mask = train_cats == level

            has_event = np.any(
                train_df.loc[mask, "Event"].to_numpy() == 1
            )

            if has_event:
                valid_levels.append(level)
            else:
                censored_levels.append(level)

        # ------------------------------------------------------
        # Match R:
        #
        # if (length(valid_levels) < 2) skip column
        # ------------------------------------------------------

        if len(valid_levels) < 2:

            metadata[col] = {
                "valid_levels": valid_levels,
                "censored_levels": censored_levels,
                "baseline": None,
                "kept_levels": [],
                "skipped": True,
            }

            continue

        # ------------------------------------------------------
        # First valid training level is baseline
        # ------------------------------------------------------

        baseline = valid_levels[0]

        kept_levels = valid_levels[1:]

        # ------------------------------------------------------
        # Encode
        # ------------------------------------------------------

        X_train_col = np.zeros(
            (len(train_df), len(kept_levels)),
            dtype=np.float32,
        )

        X_test_col = np.zeros(
            (len(test_df), len(kept_levels)),
            dtype=np.float32,
        )

        level_to_index = {
            level: j
            for j, level in enumerate(kept_levels)
        }

        # Training
        for i, level in enumerate(train_cats):

            if level in level_to_index:
                X_train_col[
                    i,
                    level_to_index[level]
                ] = 1.0

            # baseline / censored -> all zero

        # Test
        for i, level in enumerate(test_cats):

            if level in level_to_index:
                X_test_col[
                    i,
                    level_to_index[level]
                ] = 1.0

            # baseline / censored / unseen -> all zero

        train_blocks.append(X_train_col)
        test_blocks.append(X_test_col)

        metadata[col] = {
            "valid_levels": valid_levels,
            "censored_levels": censored_levels,
            "baseline": baseline,
            "kept_levels": kept_levels,
            "skipped": False,
        }

    # ----------------------------------------------------------
    # Combine categorical variables
    # ----------------------------------------------------------

    if len(train_blocks) == 0:

        X_train = np.zeros(
            (len(train_df), 0),
            dtype=np.float32,
        )

        X_test = np.zeros(
            (len(test_df), 0),
            dtype=np.float32,
        )

    else:

        X_train = np.concatenate(
            train_blocks,
            axis=1,
        )

        X_test = np.concatenate(
            test_blocks,
            axis=1,
        )

    return (
        X_train,
        X_test,
        metadata,
    )


def build_ohe_arrays(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cat_cols: List[str],
) -> Tuple[
    np.ndarray,
    np.ndarray,
    OneHotEncoder,
]:

    if len(cat_cols) == 0:
        raise ValueError(
            "No categorical columns were supplied to OHE."
        )

    encoder = make_one_hot_encoder()


    X_train = encoder.fit_transform(

        train_df[
            cat_cols
        ]
    ).astype(
        "float32"
    )


    X_test = encoder.transform(

        test_df[
            cat_cols
        ]
    ).astype(
        "float32"
    )


    if X_train.shape[1] == 0:

        raise ValueError(

            "OHE produced zero columns. "
            "The training split may not contain usable "
            "categorical variation."
        )


    return (
        X_train,
        X_test,
        encoder,
    )


# ============================================================
# 7. CONTINUOUS VARIABLES
# ============================================================

def build_cont_arrays(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cont_cols: List[str],
) -> Tuple[
    np.ndarray,
    np.ndarray,
]:

    scaler = StandardScaler()


    X_train = scaler.fit_transform(

        train_df[
            cont_cols
        ]

    ).astype(
        "float32"
    )


    X_test = scaler.transform(

        test_df[
            cont_cols
        ]

    ).astype(
        "float32"
    )


    return (
        X_train,
        X_test,
    )


# ============================================================
# 8. CALL YOUR EXISTING R KM ENCODER
# ============================================================

def run_r_km_encoder(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    work_dir: str,
) -> Tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    List[str],
]:

    """
    Calls your existing rcode_full_km.R.

    Expected R outputs:

        X_train_km_greedy.csv
        X_test_km_greedy.csv
        category_km_profiles_greedy.csv

        X_train_km_full.csv
        X_test_km_full.csv
        category_km_profiles_full.csv
    """


    if not os.path.exists(
        R_ENCODER_SCRIPT
    ):

        raise FileNotFoundError(

            "\nR encoder script not found:\n"
            f"{R_ENCODER_SCRIPT}\n\n"
            "Change R_ENCODER_SCRIPT at the top "
            "of the Python file."
        )


    os.makedirs(
        work_dir,
        exist_ok=True,
    )


    train_csv = os.path.join(
        work_dir,
        "train.csv",
    )

    test_csv = os.path.join(
        work_dir,
        "test.csv",
    )

    encoded_dir = os.path.join(
        work_dir,
        "encoded",
    )


    # --------------------------------------------------------
    # Clear previous encoded results
    # --------------------------------------------------------

    if os.path.exists(
        encoded_dir
    ):

        shutil.rmtree(
            encoded_dir
        )


    os.makedirs(
        encoded_dir,
        exist_ok=True,
    )


    # --------------------------------------------------------
    # Save train/test data for R
    # --------------------------------------------------------

    train_df.to_csv(
        train_csv,
        index=False,
    )


    test_df.to_csv(
        test_csv,
        index=False,
    )


    # --------------------------------------------------------
    # Call R
    # --------------------------------------------------------

    command = [

        "Rscript",

        R_ENCODER_SCRIPT,

        train_csv,

        test_csv,

        encoded_dir,
    ]


    print("\nRunning R KM encoder...")

    print(
        " ".join(
            command
        )
    )


    subprocess.run(
        command,
        check=True,
    )


    # --------------------------------------------------------
    # Expected output paths
    # --------------------------------------------------------

    greedy_train_path = os.path.join(
        encoded_dir,
        "X_train_km_greedy.csv",
    )

    greedy_test_path = os.path.join(
        encoded_dir,
        "X_test_km_greedy.csv",
    )

    greedy_profiles_path = os.path.join(
        encoded_dir,
        "category_km_profiles_greedy.csv",
    )


    full_train_path = os.path.join(
        encoded_dir,
        "X_train_km_full.csv",
    )

    full_test_path = os.path.join(
        encoded_dir,
        "X_test_km_full.csv",
    )

    full_profiles_path = os.path.join(
        encoded_dir,
        "category_km_profiles_full.csv",
    )


    expected_files = [

        greedy_train_path,
        greedy_test_path,
        greedy_profiles_path,

        full_train_path,
        full_test_path,
        full_profiles_path,
    ]


    missing = [

        path

        for path
        in expected_files

        if not os.path.exists(path)
    ]


    if len(missing) > 0:

        raise FileNotFoundError(

            "\nR encoder finished but these output "
            "files were not produced:\n\n"

            + "\n".join(missing)
        )


    # ========================================================
    # Load greedy representation
    # ========================================================

    X_km_greedy_train = (

        pd.read_csv(
            greedy_train_path
        )

        .values

        .astype(
            "float32"
        )
    )


    X_km_greedy_test = (

        pd.read_csv(
            greedy_test_path
        )

        .values

        .astype(
            "float32"
        )
    )


    greedy_profiles_df = pd.read_csv(

        greedy_profiles_path,

        index_col=0,
    )


    X_km_greedy_cat_profiles = (

        greedy_profiles_df

        .values

        .astype(
            "float32"
        )
    )


    # ========================================================
    # Load full KM representation
    # ========================================================

    X_km_full_train = (

        pd.read_csv(
            full_train_path
        )

        .values

        .astype(
            "float32"
        )
    )


    X_km_full_test = (

        pd.read_csv(
            full_test_path
        )

        .values

        .astype(
            "float32"
        )
    )


    full_profiles_df = pd.read_csv(

        full_profiles_path,

        index_col=0,
    )


    X_km_full_cat_profiles = (

        full_profiles_df

        .values

        .astype(
            "float32"
        )
    )


    # ========================================================
    # Category labels
    # ========================================================

    greedy_labels = list(
        greedy_profiles_df.index
    )


    full_labels = list(
        full_profiles_df.index
    )


    if greedy_labels != full_labels:

        raise ValueError(

            "Greedy and full KM category "
            "ordering differs."
        )


    km_cat_labels = full_labels


    # ========================================================
    # Dimension checks
    # ========================================================

    if (
        X_km_greedy_train.shape[0]
        != len(train_df)
    ):

        raise ValueError(

            "KM greedy train rows do not match "
            "training dataset."
        )


    if (
        X_km_greedy_test.shape[0]
        != len(test_df)
    ):

        raise ValueError(

            "KM greedy test rows do not match "
            "test dataset."
        )


    if (
        X_km_full_train.shape[0]
        != len(train_df)
    ):

        raise ValueError(

            "KM full train rows do not match "
            "training dataset."
        )


    if (
        X_km_full_test.shape[0]
        != len(test_df)
    ):

        raise ValueError(

            "KM full test rows do not match "
            "test dataset."
        )


    if (
        X_km_greedy_train.shape[1]
        !=
        X_km_greedy_test.shape[1]
    ):

        raise ValueError(

            "KM greedy train/test "
            "dimensions differ."
        )


    if (
        X_km_full_train.shape[1]
        !=
        X_km_full_test.shape[1]
    ):

        raise ValueError(

            "KM full train/test "
            "dimensions differ."
        )


    if (
        X_km_greedy_cat_profiles.shape[1]
        !=
        X_km_greedy_train.shape[1]
    ):

        raise ValueError(

            "KM greedy category profile "
            "dimension does not match "
            "KM greedy patient representation."
        )


    if (
        X_km_full_cat_profiles.shape[1]
        !=
        X_km_full_train.shape[1]
    ):

        raise ValueError(

            "KM full category profile "
            "dimension does not match "
            "KM full patient representation."
        )


    print(
        "KM greedy dimension:",
        X_km_greedy_train.shape[1],
    )

    print(
        "KM full dimension:",
        X_km_full_train.shape[1],
    )


    return (

        X_km_greedy_train,

        X_km_greedy_test,

        X_km_greedy_cat_profiles,

        X_km_full_train,

        X_km_full_test,

        X_km_full_cat_profiles,

        km_cat_labels,
    )


# ============================================================
# 9. COX LOSS
# ============================================================

# ============================================================
# 9. COX LOSS — EFRON TIES
# Based on bydmitry's TensorFlow Efron implementation
# ============================================================

def negative_cox_partial_log_likelihood(
    time,
    event,
    risk_score,
):
    time = tf.reshape(tf.cast(time, tf.float32), [-1])
    event = tf.reshape(tf.cast(event, tf.float32), [-1])
    risk = tf.reshape(tf.cast(risk_score, tf.float32), [-1])

    # Sort by descending time
    order = tf.argsort(
        time,
        direction="DESCENDING",
    )

    time = tf.gather(time, order)
    event = tf.gather(event, order)
    risk = tf.gather(risk, order)

    exp_risk = tf.exp(risk)

    # Group equal times
    _, group_ids = tf.unique(time)

    # Sum exp(risk) for each time group
    group_exp_sum = tf.math.segment_sum(
        exp_risk,
        group_ids,
    )

    # Risk set sum at each time
    risk_set_sum = tf.cumsum(
        group_exp_sum
    )

    # Number of events at each time
    event_count = tf.math.segment_sum(
        tf.cast(event, tf.int32),
        group_ids,
    )

    # Sum of risk scores for events at each time
    event_risk_sum = tf.math.segment_sum(
        risk * event,
        group_ids,
    )

    # Sum exp(risk) for tied events
    event_exp_sum = tf.math.segment_sum(
        exp_risk * event,
        group_ids,
    )

    # Keep only times where an event occurred
    mask = event_count > 0

    d = tf.boolean_mask(
        event_count,
        mask,
    )

    risk_set_sum = tf.boolean_mask(
        risk_set_sum,
        mask,
    )

    event_risk_sum = tf.boolean_mask(
        event_risk_sum,
        mask,
    )

    event_exp_sum = tf.boolean_mask(
        event_exp_sum,
        mask,
    )

    # Efron fractions:
    # 0/d, 1/d, ..., (d-1)/d
    l = tf.cast(
        tf.ragged.range(
            tf.zeros_like(d),
            d,
        ).flat_values,
        tf.float32,
    )

    d_float = tf.cast(
        d,
        tf.float32,
    )

    d_rep = tf.repeat(
        d_float,
        d,
    )

    risk_set_rep = tf.repeat(
        risk_set_sum,
        d,
    )

    event_exp_rep = tf.repeat(
        event_exp_sum,
        d,
    )

    denominator = (
        risk_set_rep
        - (l / d_rep) * event_exp_rep
    )

    log_likelihood = (
        tf.reduce_sum(event_risk_sum)
        -
        tf.reduce_sum(
            tf.math.log(denominator)
        )
    )

    return (
        -log_likelihood
        /
        (tf.reduce_sum(event) + 1e-8)
    )


# ============================================================
# 10. JOE ENCODER
# ============================================================

class SmallEncoder(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        hidden_units: List[int],
        last_activation: str = "sigmoid",
        **kwargs,
    ):

        super().__init__(
            **kwargs
        )


        activations = (

            ["sigmoid"]
            * (
                len(hidden_units)
                - 1
            )

            + [
                last_activation
            ]
        )


        self.encoder_layers = [

            tf.keras.layers.Dense(
                units,
                activation=activation,
            )

            for units, activation
            in zip(
                hidden_units,
                activations,
            )
        ]


    def call(
        self,
        x,
        training=False,
    ):

        h = x


        for layer in self.encoder_layers:

            h = layer(
                h,
                training=training,
            )


        return h


# ============================================================
# 11. NEURAL COX MODEL
# ============================================================

class NeuralCox(
    tf.keras.Model
):

    def __init__(
        self,
        encoder,
        cat_input_dim: int,
        cont_dim: int,
        **kwargs,
    ):

        super().__init__(
            **kwargs
        )


        self.encoder = encoder

        self.cat_input_dim = (
            cat_input_dim
        )

        self.cont_dim = (
            cont_dim
        )


        self.risk_layer = (

            tf.keras.layers.Dense(
                1,
                activation=None,
            )
        )


    def call(
        self,
        inputs,
        training=False,
    ):

        cat_x, cont_x = inputs


        if self.encoder is None:

            encoded_cat = cat_x

        else:

            encoded_cat = self.encoder(

                cat_x,

                training=training,
            )


        combined = tf.concat(

            [
                encoded_cat,
                cont_x,
            ],

            axis=1,
        )


        return self.risk_layer(
            combined
        )


    def encode(
        self,
        X_cat: np.ndarray,
    ) -> np.ndarray:

        if self.encoder is None:

            return X_cat


        return self.encoder(

            tf.constant(
                X_cat,
                dtype=tf.float32,
            ),

            training=False,

        ).numpy()


# ============================================================
# 12. TRAIN NEURAL COX
# ============================================================

def train_neural_cox(
    model: NeuralCox,
    X_cat_train: np.ndarray,
    X_cont_train: np.ndarray,
    time_train: np.ndarray,
    event_train: np.ndarray,
    epochs: int = EPOCHS,
    lr: float = LR,
    verbose: int = 1,
) -> NeuralCox:

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=lr
    )

    # IMPORTANT for ReduceLROnPlateau in custom loop
    model.optimizer = optimizer

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="loss",
        factor=0.5,
        patience=20,
        verbose=1,
        mode="min",
        min_delta=1e-4,
        cooldown=0,
        min_lr=1e-5,
    )

    reduce_lr.set_model(model)
    reduce_lr.on_train_begin()

    cat_tensor = tf.constant(
        X_cat_train,
        dtype=tf.float32,
    )

    cont_tensor = tf.constant(
        X_cont_train,
        dtype=tf.float32,
    )

    time_tensor = tf.constant(
        time_train,
        dtype=tf.float32,
    )

    event_tensor = tf.constant(
        event_train,
        dtype=tf.float32,
    )

    for epoch in range(epochs):

        reduce_lr.on_epoch_begin(epoch)

        with tf.GradientTape() as tape:

            risk = model(
                [
                    cat_tensor,
                    cont_tensor,
                ],
                training=True,
            )

            loss = negative_cox_partial_log_likelihood(
                time_tensor,
                event_tensor,
                risk,
            )

        gradients = tape.gradient(
            loss,
            model.trainable_variables,
        )

        gradient_variable_pairs = [
            (gradient, variable)
            for gradient, variable
            in zip(
                gradients,
                model.trainable_variables,
            )
            if gradient is not None
        ]

        optimizer.apply_gradients(
            gradient_variable_pairs
        )

        current_loss = float(
            loss.numpy()
        )

        # Give the loss to ReduceLROnPlateau
        reduce_lr.on_epoch_end(
            epoch,
            logs={
                "loss": current_loss,
            },
        )

        # Read LR AFTER ReduceLROnPlateau potentially changed it
        current_lr = float(
            tf.keras.backend.get_value(
                optimizer.learning_rate
            )
        )

        if (
            verbose
            and (
                epoch == 0
                or (epoch + 1) % 50 == 0
            )
        ):
            print(
                f"  epoch={epoch + 1:04d}  "
                f"loss={current_loss:.5f}  "
                f"lr={current_lr:.7f}"
            )

    reduce_lr.on_train_end()

    return model


# ============================================================
# 13. PREDICT RISK
# ============================================================

def predict_risk(
    model: NeuralCox,
    X_cat: np.ndarray,
    X_cont: np.ndarray,
) -> np.ndarray:

    prediction = model(

        [

            tf.constant(
                X_cat,
                dtype=tf.float32,
            ),

            tf.constant(
                X_cont,
                dtype=tf.float32,
            ),
        ],

        training=False,
    )


    return (
        prediction
        .numpy()
        .reshape(-1)
    )


# ============================================================
# 14. SURVIVAL ARRAY
# ============================================================

def make_survival_array(
    event: np.ndarray,
    time: np.ndarray,
) -> np.ndarray:

    return np.array(

        list(

            zip(

                event.astype(bool),

                time.astype(float),
            )
        ),

        dtype=[
            ("event", bool),
            ("time", float),
        ],
    )


# ============================================================
# 15. IBS TIME GRID
# ============================================================

def make_ibs_time_grid(
    time_train: np.ndarray,
    time_test: np.ndarray,
    n_times: int = 100,
) -> np.ndarray:

    time_train = np.asarray(
        time_train,
        dtype=float,
    )


    time_test = np.asarray(
        time_test,
        dtype=float,
    )


    # --------------------------------------------------------
    # Stay inside both train and test support.
    # --------------------------------------------------------

    lower = max(

        np.min(
            time_test
        ),

        1e-8,
    )


    upper = min(

        np.max(
            time_test
        ),

        np.max(
            time_train
        ),
    )


    epsilon = (

        1e-8

        * max(
            1.0,
            upper,
        )
    )


    lower = lower + epsilon

    upper = upper - epsilon


    if (

        not np.isfinite(lower)

        or

        not np.isfinite(upper)

        or

        upper <= lower
    ):

        return np.array(
            [],
            dtype=float,
        )


    return np.linspace(

        lower,

        upper,

        n_times,
    )


# ============================================================
# 16. COX SURVIVAL PROBABILITIES
# ============================================================

def estimate_survival_probabilities_from_cox_risk(
    train_risk: np.ndarray,
    test_risk: np.ndarray,
    event_train: np.ndarray,
    time_train: np.ndarray,
    times: np.ndarray,
) -> np.ndarray:

    breslow = (
        BreslowEstimator()
    )


    breslow.fit(

        linear_predictor=np.asarray(
            train_risk,
            dtype=float,
        ),

        event=np.asarray(
            event_train,
            dtype=bool,
        ),

        time=np.asarray(
            time_train,
            dtype=float,
        ),
    )


    survival_functions = (
        breslow.get_survival_function(

            np.asarray(
                test_risk,
                dtype=float,
            )
        )
    )


    survival_probabilities = np.asarray(

        [

            survival_function(
                times
            )

            for survival_function
            in survival_functions
        ],

        dtype=float,
    )


    return np.clip(

        survival_probabilities,

        1e-8,

        1.0,
    )


# ============================================================
# 17. IBS
# ============================================================

def compute_integrated_brier_score_for_model(
    model: NeuralCox,
    X_cat_train: np.ndarray,
    X_cont_train: np.ndarray,
    X_cat_test: np.ndarray,
    X_cont_test: np.ndarray,
    time_train: np.ndarray,
    event_train: np.ndarray,
    time_test: np.ndarray,
    event_test: np.ndarray,
    n_times: int = 100,
) -> float:
    """
    Mirrors the R evaluation:

        time_interest <- unique(seq(
            quantile(df_test$Time, 0.05, na.rm = TRUE),
            quantile(df_test$Time, 0.95, na.rm = TRUE),
            length.out = 100
        ))

        Score(
            ...,
            cens.model = "km",
            cens.data = df_train,
            times = time_interest,
            summary = "ibs"
        )

    Evaluation times are based on the 5th-95th percentile of
    TEST follow-up, while IPCW censoring is estimated from TRAIN.
    """

    # np.nanquantile corresponds to R's quantile(..., na.rm = TRUE)
    evaluation_times = np.unique(
        np.linspace(
            np.nanquantile(time_test, 0.05),
            np.nanquantile(time_test, 0.95),
            n_times,
        )
    )

    if len(evaluation_times) < 2:
        return float("nan")

    y_train = make_survival_array(
        event_train,
        time_train,
    )

    y_test = make_survival_array(
        event_test,
        time_test,
    )

    train_risk = predict_risk(
        model,
        X_cat_train,
        X_cont_train,
    )

    test_risk = predict_risk(
        model,
        X_cat_test,
        X_cont_test,
    )

    survival_probabilities = (
        estimate_survival_probabilities_from_cox_risk(
            train_risk=train_risk,
            test_risk=test_risk,
            event_train=event_train,
            time_train=time_train,
            times=evaluation_times,
        )
    )

    return float(
        integrated_brier_score(
            y_train,
            y_test,
            survival_probabilities,
            evaluation_times,
        )
    )


# ============================================================
# 18. SCORE MODEL
# ============================================================

def score_model(
    model: NeuralCox,
    X_cat_train: np.ndarray,
    X_cont_train: np.ndarray,
    X_cat_test: np.ndarray,
    X_cont_test: np.ndarray,
    time_train: np.ndarray,
    event_train: np.ndarray,
    time_test: np.ndarray,
    event_test: np.ndarray,
) -> Dict[str, float]:

    test_risk = predict_risk(

        model,

        X_cat_test,

        X_cont_test,
    )


    cindex_result = (
        concordance_index_censored(

            event_test.astype(bool),

            time_test,

            test_risk,
        )
    )


    cindex = float(
        cindex_result[0]
    )


    try:

        ibs = (
            compute_integrated_brier_score_for_model(

                model=model,

                X_cat_train=
                    X_cat_train,

                X_cont_train=
                    X_cont_train,

                X_cat_test=
                    X_cat_test,

                X_cont_test=
                    X_cont_test,

                time_train=
                    time_train,

                event_train=
                    event_train,

                time_test=
                    time_test,

                event_test=
                    event_test,
            )
        )


    except Exception as error:

        print(

            f"  WARNING: IBS could not "
            f"be calculated for "
            f"{model.name}.\n"

            f"  {error}"
        )

        ibs = float(
            "nan"
        )


    return {

        "cindex":
            cindex,

        "ibs":
            ibs,
    }


# ============================================================
# 19. TRAIN + SCORE CONVENIENCE FUNCTION
# ============================================================

def fit_and_score_model(
    model: NeuralCox,
    X_cat_train: np.ndarray,
    X_cont_train: np.ndarray,
    X_cat_test: np.ndarray,
    X_cont_test: np.ndarray,
    time_train: np.ndarray,
    event_train: np.ndarray,
    time_test: np.ndarray,
    event_test: np.ndarray,
    epochs: int,
    lr: float,
) -> Dict[str, float]:

    train_neural_cox(

        model=model,

        X_cat_train=
            X_cat_train,

        X_cont_train=
            X_cont_train,

        time_train=
            time_train,

        event_train=
            event_train,

        epochs=epochs,

        lr=lr,
    )


    return score_model(

        model=model,

        X_cat_train=
            X_cat_train,

        X_cont_train=
            X_cont_train,

        X_cat_test=
            X_cat_test,

        X_cont_test=
            X_cont_test,

        time_train=
            time_train,

        event_train=
            event_train,

        time_test=
            time_test,

        event_test=
            event_test,
    )


# ============================================================
# 20. RUN ONE REAL-DATA SPLIT
# ============================================================

def run_one_real_experiment(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cont_cols: List[str],
    cat_cols: List[str],
    dataset_name: str,
    seed: int,
    out_dir: str,
    encoding_dim: int = ENCODING_DIM,
    epochs: int = EPOCHS,
    lr: float = LR,
) -> Dict:

    set_seed(
        seed
    )


    # Clear old TF graph/models.
    tf.keras.backend.clear_session()


    # ========================================================
    # Outcomes
    # ========================================================

    time_train = (

        train_df["Time"]

        .to_numpy(
            dtype=float
        )
    )


    event_train = (

        train_df["Event"]

        .to_numpy(
            dtype=int
        )
    )


    time_test = (

        test_df["Time"]

        .to_numpy(
            dtype=float
        )
    )


    event_test = (

        test_df["Event"]

        .to_numpy(
            dtype=int
        )
    )


    if event_train.sum() == 0:

        raise ValueError(
            "Training split contains no events."
        )


    if event_test.sum() == 0:

        raise ValueError(
            "Test split contains no events."
        )


    # ========================================================
    # Continuous covariates
    # ========================================================

    (
        X_cont_train,
        X_cont_test,

    ) = build_cont_arrays(

        train_df,

        test_df,

        cont_cols,
    )


    cont_dim = int(
        X_cont_train.shape[1]
    )


    # ========================================================
    # OHE
    # ========================================================

    (
        X_ohe_train,
        X_ohe_test,
        ohe_encoder,

    ) = build_ohe_arrays_r_style(

        train_df,

        test_df,

        cat_cols,
    )


    ohe_dim = int(
        X_ohe_train.shape[1]
    )


    # ========================================================
    # KM
    # ========================================================

    r_work_dir = os.path.join(

        out_dir,

        "r_km_cache",

        f"seed_{seed:03d}",
    )


    (
        X_km_greedy_train,

        X_km_greedy_test,

        X_km_greedy_cat_profiles,

        X_km_full_train,

        X_km_full_test,

        X_km_full_cat_profiles,

        km_cat_labels,

    ) = run_r_km_encoder(

        train_df=train_df,

        test_df=test_df,

        work_dir=r_work_dir,
    )


    km_greedy_dim = int(

        X_km_greedy_train.shape[1]
    )


    km_full_dim = int(

        X_km_full_train.shape[1]
    )


    if km_greedy_dim == 0:

        raise ValueError(

            "R KM encoder selected zero "
            "greedy KM variables."
        )


    if km_full_dim == 0:

        raise ValueError(

            "R KM encoder returned zero "
            "full KM variables."
        )


    # ========================================================
    # Metadata
    # ========================================================

    train_category_levels = {
        col: set(
            train_df[col].astype(str)
        )
        for col in cat_cols
    }

    test_category_levels = {
        col: set(
            test_df[col].astype(str)
        )
        for col in cat_cols
    }

    unseen_test_levels = {
        col: (
            test_category_levels[col]
            - train_category_levels[col]
        )
        for col in cat_cols
    }

    total_train_levels = sum(
        len(levels)
        for levels in train_category_levels.values()
    )

    total_test_levels = sum(
        len(levels)
        for levels in test_category_levels.values()
    )

    total_unseen_test_levels = sum(
        len(levels)
        for levels in unseen_test_levels.values()
    )


    results: Dict = {

        "dataset":
            dataset_name,

        "seed":
            seed,

        "n_train":
            len(train_df),

        "n_test":
            len(test_df),

        "n_cont":
            cont_dim,

        "n_categorical_features":
            len(cat_cols),

        "n_categories_train":
            total_train_levels,

        "n_categories_test":
            total_test_levels,

        "n_unseen_categories_test":
            total_unseen_test_levels,

        "events_train":
            int(
                event_train.sum()
            ),

        "events_test":
            int(
                event_test.sum()
            ),

        "event_fraction_train":
            float(
                event_train.mean()
            ),

        "event_fraction_test":
            float(
                event_test.mean()
            ),

        "ohe_dim":
            ohe_dim,

        "km_greedy_dim":
            km_greedy_dim,

        "km_full_dim":
            km_full_dim,

        "encoding_dim":
            encoding_dim,
    }


    # ========================================================
    # MODEL 1:
    # NN OHE
    # ========================================================

    print("\n")
    print("-" * 60)
    print("NN OHE")
    print("-" * 60)


    model_nn_ohe = NeuralCox(

        encoder=None,

        cat_input_dim=ohe_dim,

        cont_dim=cont_dim,

        name="nn_ohe",
    )


    scores = fit_and_score_model(

        model=
            model_nn_ohe,

        X_cat_train=
            X_ohe_train,

        X_cont_train=
            X_cont_train,

        X_cat_test=
            X_ohe_test,

        X_cont_test=
            X_cont_test,

        time_train=
            time_train,

        event_train=
            event_train,

        time_test=
            time_test,

        event_test=
            event_test,

        epochs=
            epochs,

        lr=
            lr,
    )


    results[
        "cindex_nn_ohe"
    ] = scores["cindex"]


    results[
        "ibs_nn_ohe"
    ] = scores["ibs"]


    print(
        f"C-index = "
        f"{scores['cindex']:.4f}"
    )

    print(
        f"IBS     = "
        f"{scores['ibs']:.4f}"
    )


    # ========================================================
    # MODEL 2-3:
    # JOECHAR
    #
    # Full fixed-grid KM representation
    # -> neural low-dimensional encoding
    # ========================================================

    joechar_variants = [

        (
            "sigmoid",

            [
                3,
                encoding_dim,
            ],

            "sigmoid",
        ),

        (
            "linear",

            [
                encoding_dim,
            ],

            "linear",
        ),
    ]


    for (
        variant,
        hidden_units,
        last_activation,

    ) in joechar_variants:


        print("\n")
        print("-" * 60)

        print(
            f"JOECHAR {variant.upper()}"
        )

        print("-" * 60)


        encoder = SmallEncoder(

            hidden_units=
                hidden_units,

            last_activation=
                last_activation,

            name=
                f"joechar_encoder_{variant}",
        )


        model = NeuralCox(

            encoder=encoder,

            cat_input_dim=
                km_full_dim,

            cont_dim=
                cont_dim,

            # Keep original result names for comparison
            name=
                f"joe_km_greedy_{variant}",
        )


        scores = fit_and_score_model(

            model=
                model,

            X_cat_train=
                X_km_full_train,

            X_cont_train=
                X_cont_train,

            X_cat_test=
                X_km_full_test,

            X_cont_test=
                X_cont_test,

            time_train=
                time_train,

            event_train=
                event_train,

            time_test=
                time_test,

            event_test=
                event_test,

            epochs=
                epochs,

            lr=
                lr,
        )


        key = (
            f"joe_km_greedy_{variant}"
        )


        results[
            f"cindex_{key}"
        ] = scores["cindex"]


        results[
            f"ibs_{key}"
        ] = scores["ibs"]


        print(
            f"C-index = "
            f"{scores['cindex']:.4f}"
        )

        print(
            f"IBS     = "
            f"{scores['ibs']:.4f}"
        )


    # ========================================================
    # MODEL 4-5:
    # JOEOH
    #
    # OHE representation
    # -> neural low-dimensional encoding
    # ========================================================

    joeoh_variants = [

        (
            "sigmoid",

            [
                3,
                encoding_dim,
            ],

            "sigmoid",
        ),

        (
            "linear",

            [
                encoding_dim,
            ],

            "linear",
        ),
    ]


    for (
        variant,
        hidden_units,
        last_activation,

    ) in joeoh_variants:


        print("\n")
        print("-" * 60)

        print(
            f"JOEOH {variant.upper()}"
        )

        print("-" * 60)


        encoder = SmallEncoder(

            hidden_units=
                hidden_units,

            last_activation=
                last_activation,

            name=
                f"joeoh_encoder_{variant}",
        )


        model = NeuralCox(

            encoder=
                encoder,

            cat_input_dim=
                ohe_dim,

            cont_dim=
                cont_dim,

            name=
                f"joe_ohe_{variant}",
        )


        scores = fit_and_score_model(

            model=
                model,

            X_cat_train=
                X_ohe_train,

            X_cont_train=
                X_cont_train,

            X_cat_test=
                X_ohe_test,

            X_cont_test=
                X_cont_test,

            time_train=
                time_train,

            event_train=
                event_train,

            time_test=
                time_test,

            event_test=
                event_test,

            epochs=
                epochs,

            lr=
                lr,
        )


        key = (
            f"joe_ohe_{variant}"
        )


        results[
            f"cindex_{key}"
        ] = scores["cindex"]


        results[
            f"ibs_{key}"
        ] = scores["ibs"]


        print(
            f"C-index = "
            f"{scores['cindex']:.4f}"
        )

        print(
            f"IBS     = "
            f"{scores['ibs']:.4f}"
        )


    # ========================================================
    # MODEL 6:
    # KM GREEDY
    # ========================================================

    print("\n")
    print("-" * 60)
    print("KM GREEDY")
    print("-" * 60)


    model_km_greedy = NeuralCox(

        encoder=None,

        cat_input_dim=
            km_greedy_dim,

        cont_dim=
            cont_dim,

        name=
            "km_greedy",
    )


    scores = fit_and_score_model(

        model=
            model_km_greedy,

        X_cat_train=
            X_km_greedy_train,

        X_cont_train=
            X_cont_train,

        X_cat_test=
            X_km_greedy_test,

        X_cont_test=
            X_cont_test,

        time_train=
            time_train,

        event_train=
            event_train,

        time_test=
            time_test,

        event_test=
            event_test,

        epochs=
            epochs,

        lr=
            lr,
    )


    results[
        "cindex_km_greedy"
    ] = scores["cindex"]


    results[
        "ibs_km_greedy"
    ] = scores["ibs"]


    print(
        f"C-index = "
        f"{scores['cindex']:.4f}"
    )

    print(
        f"IBS     = "
        f"{scores['ibs']:.4f}"
    )


    # ========================================================
    # Gains relative to NN OHE
    # ========================================================

    baseline_cindex = (
        results[
            "cindex_nn_ohe"
        ]
    )


    baseline_ibs = (
        results[
            "ibs_nn_ohe"
        ]
    )


    comparison_methods = [

        "km_greedy",

        "joe_km_greedy_sigmoid",

        "joe_km_greedy_linear",

        "joe_ohe_sigmoid",

        "joe_ohe_linear",
    ]


    for method in comparison_methods:

        # ----------------------------------------------------
        # C-index:
        # higher is better
        #
        # Positive = better than NN OHE
        # ----------------------------------------------------

        results[
            f"gain_cindex_{method}"
        ] = (

            results[
                f"cindex_{method}"
            ]

            - baseline_cindex
        )


        # ----------------------------------------------------
        # IBS:
        # lower is better
        #
        # Positive = better than NN OHE
        # ----------------------------------------------------

        results[
            f"gain_ibs_{method}"
        ] = (

            baseline_ibs

            - results[
                f"ibs_{method}"
            ]
        )


    # ========================================================
    # Clean TensorFlow memory
    # ========================================================

    del model_nn_ohe
    del model_km_greedy

    gc.collect()

    tf.keras.backend.clear_session()


    return results


# ============================================================
# 21. RUN REPEATED SPLITS FOR ONE DATASET
# ============================================================

def run_real_dataset(
    config: DatasetConfig,
    seeds=range(N_REPEATS),
    train_frac: float = TRAIN_FRAC,
    encoding_dim: int = ENCODING_DIM,
    epochs: int = EPOCHS,
    lr: float = LR,
    stratify_event: bool = STRATIFY_EVENT,
) -> pd.DataFrame:

    # ========================================================
    # Load/preprocess once
    # ========================================================

    if config.name == "CIBMTR":

        (
            full_df,
            cont_cols,
            cat_cols,

        ) = load_cibmtr_dataset(

            cibmtr_file=os.path.join(
                DATA_DIR,
                "CIBMTR.csv",
            ),

            dictionary_file=os.path.join(
                DATA_DIR,
                "data_dictionary.csv",
            ),
        )

    else:

        (
            full_df,
            cont_cols,
            cat_cols,

        ) = load_real_dataset(
            config
        )


    dataset_out_dir = os.path.join(

        RESULTS_ROOT,

        config.name,
    )


    os.makedirs(

        dataset_out_dir,

        exist_ok=True,
    )


    all_rows = []


    # ========================================================
    # Repeated train/test experiments
    # ========================================================

    for repetition, seed in enumerate(
        seeds,
        start=1,
    ):


        print("\n\n")
        print("#" * 75)

        print(
            f"DATASET: {config.name}"
        )

        print(
            f"REPETITION: {repetition}"
        )

        print(
            f"SEED: {seed}"
        )

        print("#" * 75)


        set_seed(
            seed
        )


        if stratify_event:

            stratification = (
                full_df["Event"]
            )

        else:

            stratification = None


        (
            train_df,
            test_df,

        ) = train_test_split(

            full_df,

            train_size=
                train_frac,

            random_state=
                seed,

            shuffle=
                True,

            stratify=
                stratification,
        )


        train_df = (

            train_df

            .reset_index(
                drop=True
            )
        )


        test_df = (

            test_df

            .reset_index(
                drop=True
            )
        )


        print(
            "\nTrain rows:",
            len(train_df),
        )

        print(
            "Test rows:",
            len(test_df),
        )


        print(
            "Train events:",
            int(
                train_df[
                    "Event"
                ].sum()
            ),
        )


        print(
            "Test events:",
            int(
                test_df[
                    "Event"
                ].sum()
            ),
        )


        print(
            "Train event fraction:",
            round(
                float(
                    train_df[
                        "Event"
                    ].mean()
                ),
                4,
            ),
        )


        print(
            "Test event fraction:",
            round(
                float(
                    test_df[
                        "Event"
                    ].mean()
                ),
                4,
            ),
        )


        print(
            "Categorical features:",
            len(cat_cols),
        )

        if len(cat_cols) == 1:

            print(
                "Train joint categories:",
                train_df[
                    cat_cols[0]
                ].nunique(),
            )

            print(
                "Test joint categories:",
                test_df[
                    cat_cols[0]
                ].nunique(),
            )

        else:

            print(
                "Train categorical levels by feature:"
            )

            for col in cat_cols:
                print(
                    f"  {col}: "
                    f"{train_df[col].nunique()}"
                )

            print(
                "Test categorical levels by feature:"
            )

            for col in cat_cols:
                print(
                    f"  {col}: "
                    f"{test_df[col].nunique()}"
                )


        # ====================================================
        # Run models
        # ====================================================

        try:

            result = (
                run_one_real_experiment(

                    train_df=
                        train_df,

                    test_df=
                        test_df,

                    cont_cols=
                        cont_cols,

                    cat_cols=
                        cat_cols,

                    dataset_name=
                        config.name,

                    seed=
                        seed,

                    out_dir=
                        dataset_out_dir,

                    encoding_dim=
                        encoding_dim,

                    epochs=
                        epochs,

                    lr=
                        lr,
                )
            )


            result[
                "repetition"
            ] = repetition


            result[
                "error"
            ] = ""


        except Exception as error:

            print("\n")
            print("!" * 75)

            print(
                f"ERROR IN "
                f"{config.name}, "
                f"REPETITION {repetition}"
            )

            print(
                str(error)
            )

            print("!" * 75)


            result = {

                "dataset":
                    config.name,

                "seed":
                    seed,

                "repetition":
                    repetition,

                "error":
                    str(error),
            }


        all_rows.append(
            result
        )


        # ====================================================
        # Save partial results after EVERY repetition
        #
        # Useful for Colab disconnects.
        # ====================================================

        partial_results = (
            pd.DataFrame(
                all_rows
            )
        )


        partial_results.to_csv(

            os.path.join(

                dataset_out_dir,

                f"{config.name}"
                f"_results_partial.csv",
            ),

            index=False,
        )


    # ========================================================
    # Final dataset results
    # ========================================================

    results_df = pd.DataFrame(
        all_rows
    )


    results_df.to_csv(

        os.path.join(

            dataset_out_dir,

            f"{config.name}"
            f"_results.csv",
        ),

        index=False,
    )


    return results_df


# ============================================================
# 22. SUMMARY TABLE
# ============================================================

def summarise_results(
    results: pd.DataFrame,
) -> pd.DataFrame:

    methods = [

        (
            "NN OHE",
            "nn_ohe",
        ),

        (
            "KM Greedy",
            "km_greedy",
        ),

        (
            "JOEOH 3SIG",
            "joe_ohe_sigmoid",
        ),

        (
            "JOEOH 1LIN",
            "joe_ohe_linear",
        ),

        (
            "JOECHAR 3SIG",
            "joe_km_greedy_sigmoid",
        ),

        (
            "JOECHAR 1LIN",
            "joe_km_greedy_linear",
        ),
    ]


    summary_rows = []


    for (
        display_name,
        method_key,

    ) in methods:


        cindex_column = (
            f"cindex_{method_key}"
        )


        ibs_column = (
            f"ibs_{method_key}"
        )


        if (
            cindex_column
            not in results.columns
        ):

            continue


        valid_cindex = pd.to_numeric(

            results[
                cindex_column
            ],

            errors="coerce",
        )


        valid_ibs = pd.to_numeric(

            results[
                ibs_column
            ],

            errors="coerce",
        )


        summary_rows.append(

            {

                "Method":
                    display_name,

                "C-index mean":
                    valid_cindex.mean(),

                "C-index SD":
                    valid_cindex.std(),

                "IBS mean":
                    valid_ibs.mean(),

                "IBS SD":
                    valid_ibs.std(),

                "Successful repetitions":
                    int(
                        valid_cindex
                        .notna()
                        .sum()
                    ),
            }
        )


    return pd.DataFrame(
        summary_rows
    )


# ============================================================
# 23. GAIN SUMMARY
# ============================================================

def summarise_gains(
    results: pd.DataFrame,
) -> pd.DataFrame:

    methods = [

        (
            "KM Greedy",
            "km_greedy",
        ),

        (
            "JOEOH 3SIG",
            "joe_ohe_sigmoid",
        ),

        (
            "JOEOH 1LIN",
            "joe_ohe_linear",
        ),

        (
            "JOECHAR 3SIG",
            "joe_km_greedy_sigmoid",
        ),

        (
            "JOECHAR 1LIN",
            "joe_km_greedy_linear",
        ),
    ]


    rows = []


    for (
        display_name,
        method_key,

    ) in methods:


        cindex_gain_col = (
            f"gain_cindex_{method_key}"
        )


        ibs_gain_col = (
            f"gain_ibs_{method_key}"
        )


        if (
            cindex_gain_col
            not in results.columns
        ):

            continue


        cindex_gain = pd.to_numeric(

            results[
                cindex_gain_col
            ],

            errors="coerce",
        )


        ibs_gain = pd.to_numeric(

            results[
                ibs_gain_col
            ],

            errors="coerce",
        )


        rows.append(

            {

                "Method":
                    display_name,

                "Mean C-index gain":
                    cindex_gain.mean(),

                "SD C-index gain":
                    cindex_gain.std(),

                "Mean IBS improvement":
                    ibs_gain.mean(),

                "SD IBS improvement":
                    ibs_gain.std(),
            }
        )


    return pd.DataFrame(
        rows
    )


# ============================================================
# 24. PRINT RESULT SUMMARY
# ============================================================

def print_dataset_summary(
    dataset_name: str,
    summary: pd.DataFrame,
    gain_summary: pd.DataFrame,
) -> None:

    print("\n\n")

    print("=" * 80)

    print(
        f"FINAL RESULTS: "
        f"{dataset_name}"
    )

    print("=" * 80)


    print("\nAbsolute performance:\n")


    if len(summary) > 0:

        print(

            summary.to_string(
                index=False,
            )
        )

    else:

        print(
            "No successful results."
        )


    print(
        "\n\nImprovement relative "
        "to NN OHE:\n"
    )


    if len(gain_summary) > 0:

        print(

            gain_summary.to_string(
                index=False,
            )
        )

    else:

        print(
            "No gain results."
        )


# ============================================================
# 25. MAIN
# ============================================================

if __name__ == "__main__":


    # --------------------------------------------------------
    # Basic checks
    # --------------------------------------------------------

    os.makedirs(

        RESULTS_ROOT,

        exist_ok=True,
    )


    print("\n")
    print("=" * 80)
    print("REAL DATASET JOE + KM EXPERIMENT")
    print("=" * 80)


    print(
        "\nDatasets:",
        DATASETS_TO_RUN,
    )


    print(
        "Repetitions:",
        N_REPEATS,
    )


    print(
        "Train fraction:",
        TRAIN_FRAC,
    )


    print(
        "Encoding dimension:",
        ENCODING_DIM,
    )


    print(
        "Epochs:",
        EPOCHS,
    )


    print(
        "Learning rate:",
        LR,
    )


    print(
        "Stratified splits:",
        STRATIFY_EVENT,
    )


    # ========================================================
    # Run each dataset
    # ========================================================

    combined_results = []

    combined_summaries = []

    combined_gain_summaries = []


    for dataset_name in DATASETS_TO_RUN:


        dataset_name = (
            dataset_name
            .strip()
            .upper()
        )


        if (
            dataset_name
            not in DATASETS
        ):

            print(

                f"\nSkipping unknown dataset: "
                f"{dataset_name}"
            )

            continue


        config = DATASETS[
            dataset_name
        ]


        print("\n\n")
        print("*" * 80)

        print(
            f"STARTING DATASET: "
            f"{dataset_name}"
        )

        print("*" * 80)


        try:

            # =================================================
            # Repeated experiments
            # =================================================

            results = run_real_dataset(

                config=config,

                seeds=range(
                    N_REPEATS
                ),

                train_frac=
                    TRAIN_FRAC,

                encoding_dim=
                    ENCODING_DIM,

                epochs=
                    EPOCHS,

                lr=
                    LR,

                stratify_event=
                    STRATIFY_EVENT,
            )


            # =================================================
            # Summary
            # =================================================

            summary = (
                summarise_results(
                    results
                )
            )


            gain_summary = (
                summarise_gains(
                    results
                )
            )


            print_dataset_summary(

                dataset_name,

                summary,

                gain_summary,
            )


            # =================================================
            # Save summaries
            # =================================================

            dataset_output_dir = os.path.join(

                RESULTS_ROOT,

                dataset_name,
            )


            summary.to_csv(

                os.path.join(

                    dataset_output_dir,

                    f"{dataset_name}"
                    f"_summary.csv",
                ),

                index=False,
            )


            gain_summary.to_csv(

                os.path.join(

                    dataset_output_dir,

                    f"{dataset_name}"
                    f"_gain_summary.csv",
                ),

                index=False,
            )


            # =================================================
            # Combined outputs
            # =================================================

            combined_results.append(
                results
            )


            summary_for_combined = (
                summary.copy()
            )


            summary_for_combined.insert(

                0,

                "Dataset",

                dataset_name,
            )


            combined_summaries.append(
                summary_for_combined
            )


            gain_for_combined = (
                gain_summary.copy()
            )


            gain_for_combined.insert(

                0,

                "Dataset",

                dataset_name,
            )


            combined_gain_summaries.append(
                gain_for_combined
            )


        except Exception as error:

            print("\n")
            print("!" * 80)

            print(
                f"DATASET FAILED: "
                f"{dataset_name}"
            )

            print(
                str(error)
            )

            print("!" * 80)


    # ========================================================
    # Combined raw results
    # ========================================================

    if len(
        combined_results
    ) > 0:


        combined_results_df = pd.concat(

            combined_results,

            ignore_index=True,
        )


        combined_results_df.to_csv(

            os.path.join(

                RESULTS_ROOT,

                "all_real_datasets_results.csv",
            ),

            index=False,
        )


    # ========================================================
    # Combined absolute-performance summary
    # ========================================================

    if len(
        combined_summaries
    ) > 0:


        combined_summary_df = pd.concat(

            combined_summaries,

            ignore_index=True,
        )


        combined_summary_df.to_csv(

            os.path.join(

                RESULTS_ROOT,

                "all_real_datasets_summary.csv",
            ),

            index=False,
        )


        print("\n\n")
        print("=" * 80)

        print(
            "COMBINED PERFORMANCE SUMMARY"
        )

        print("=" * 80)

        print(

            combined_summary_df
            .to_string(
                index=False
            )
        )


    # ========================================================
    # Combined gain summary
    # ========================================================

    if len(
        combined_gain_summaries
    ) > 0:


        combined_gain_df = pd.concat(

            combined_gain_summaries,

            ignore_index=True,
        )


        combined_gain_df.to_csv(

            os.path.join(

                RESULTS_ROOT,

                "all_real_datasets_gain_summary.csv",
            ),

            index=False,
        )


        print("\n\n")
        print("=" * 80)

        print(
            "COMBINED GAINS RELATIVE TO NN OHE"
        )

        print("=" * 80)

        print(

            combined_gain_df
            .to_string(
                index=False
            )
        )


    print("\n\n")
    print("=" * 80)
    print("ALL REQUESTED REAL-DATA EXPERIMENTS FINISHED")
    print("=" * 80)

    print(
        f"\nResults saved in:\n"
        f"{os.path.abspath(RESULTS_ROOT)}"
    )

Streaming output truncated to the last 5000 lines.
  epoch=0350  loss=7.73925  lr=0.0000977

Epoch 365: ReduceLROnPlateau reducing learning rate to 4.882812572759576e-05.

Epoch 385: ReduceLROnPlateau reducing learning rate to 2.441406286379788e-05.
  epoch=0400  loss=7.73925  lr=0.0000244

Epoch 405: ReduceLROnPlateau reducing learning rate to 1.220703143189894e-05.

Epoch 425: ReduceLROnPlateau reducing learning rate to 1e-05.
  epoch=0450  loss=7.73924  lr=0.0000100
  epoch=0500  loss=7.73924  lr=0.0000100
C-index = 0.5928
IBS     = 0.2014


------------------------------------------------------------
JOECHAR LINEAR
------------------------------------------------------------
  epoch=0001  loss=8.99961  lr=0.0500000
  epoch=0050  loss=7.74355  lr=0.0500000
  epoch=0100  loss=7.73925  lr=0.0500000

Epoch 128: ReduceLROnPlateau reducing learning rate to 0.02500000037252903.
  epoch=0150  loss=7.73908  lr=0.0250000

Epoch 157: ReduceLROnPlateau reducing learning rate to 0.0125000001862

In [4]:
def plot_metric_gain_boxplots(
    df_res: pd.DataFrame,
    out_dir: str,
    k_col: str = "K",
) -> None:
    """Plot gain distributions grouped by K.

    Within each K group, one boxplot is drawn for every available method.
    Vertical dashed lines separate consecutive K groups.

    C-index gain:
        method C-index - NN OHE C-index

    IBS gain:
        NN OHE IBS - method IBS

    Positive values therefore indicate improvement over NN OHE.
    """
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    method_order = [
        "km_greedy",
        "joe_km_greedy_sigmoid",
        "joe_km_greedy_linear",
        "joe_ohe_sigmoid",
        "joe_ohe_linear",
    ]

    method_labels = {
        "km_greedy": "KM",
        "joe_km_greedy_sigmoid": "JOECHAR_3SIG",
        "joe_km_greedy_linear": "JOECHAR_1LIN",
        "joe_ohe_sigmoid": "JOEOH_3SIG",
        "joe_ohe_linear": "JOEOH_1LIN",
    }

    plot_specs = [
        (
            "gain_cindex",
            "C-index gain\n(method - NN OHE)",
            "cindex_gain_boxplot.pdf",
        ),
        (
            "gain_ibs",
            "IBS gain\n(NN OHE - method)",
            "ibs_gain_boxplot.pdf",
        ),
    ]

    if k_col not in df_res.columns:
        # The `K` column is not present in the results for real datasets.
        # Instead of grouping by K, we'll plot all results together.
        k_values = [None] # Treat as a single group
        df_res_with_k = df_res.copy()
        df_res_with_k[k_col] = None
    else:
        k_values = sorted(df_res[k_col].dropna().unique())
        df_res_with_k = df_res.copy()

    if not k_values:
        print("  No K values found; skipping gain plots.")
        return

    for prefix, ylabel, filename in plot_specs:
        all_data = []
        positions = []
        tick_labels = []

        group_centres = []
        group_labels = []
        separator_positions = []

        current_position = 1

        for group_index, k_value in enumerate(k_values):
            if k_value is None:
                df_k = df_res_with_k
            else:
                df_k = df_res_with_k.loc[df_res_with_k[k_col] == k_value]

            group_positions = []

            for method in method_order:
                column = f"{prefix}_{method}"

                if column not in df_k.columns:
                    continue

                values = (
                    df_k[column]
                    .replace([np.inf, -np.inf], np.nan)
                    .dropna()
                    .to_numpy()
                )

                if len(values) == 0:
                    continue

                all_data.append(values)
                positions.append(current_position)
                tick_labels.append(method_labels[method])
                group_positions.append(current_position)

                current_position += 1

            # Record the centre of this K group for the upper group label.
            if group_positions:
                group_centres.append(np.mean(group_positions))
                if k_value is None:
                    group_labels.append(" ")
                else:
                    group_labels.append(f"K = {k_value}")

                # Separator goes midway between this group and the next.
                separator_positions.append(current_position - 0.5)

                # Add a small blank space between K groups.
                current_position += 1

        if not all_data:
            print(f"  No valid data found for {prefix}; skipping gain plot.")
            continue

        # The final separator would appear after the last group, so remove it.
        separator_positions = separator_positions[:-1]

        figure_width = max(10, 0.9 * len(all_data) + 2.5)

        fig, ax = plt.subplots(
            figsize=(figure_width, 5.2)
        )

        boxplot = ax.boxplot(
            all_data,
            positions=positions,
            widths=0.65,
            patch_artist=True,
            showmeans=False,
            showfliers=False,
        )

        for box in boxplot["boxes"]:
            box.set(
                facecolor="lightgrey",
                edgecolor="black",
                alpha=1.0,
            )

        for whisker in boxplot["whiskers"]:
            whisker.set(color="black")

        for cap in boxplot["caps"]:
            cap.set(color="black")

        for median in boxplot["medians"]:
            median.set(
                color="black",
                linewidth=2,
            )

        for flier in boxplot["fliers"]:
            flier.set(
                marker="o",
                markerfacecolor="none",
                markeredgecolor="black",
                markersize=4,
            )

        # Baseline: no difference from NN OHE.
        ax.axhline(
            0,
            color="grey",
            linewidth=0.8,
            linestyle="--",
            zorder=0,
        )

        # Separate the K groups.
        for separator in separator_positions:
            ax.axvline(
                separator,
                color="grey",
                linewidth=0.8,
                linestyle="--",
                alpha=0.8,
                zorder=0,
            )





        ax.set_xticks(positions)
        ax.set_xticklabels(
            tick_labels,
            rotation=30,
            ha="right",
            fontsize=15,
        )

        # Y-axis label
        ax.set_ylabel(
            ylabel,
            fontsize=15,
        )

        # Y-axis tick numbers
        ax.tick_params(
            axis="y",
            labelsize=15,
        )

        # Add a second x-axis above the plot containing the K labels.
        if len(k_values) > 1 or k_values[0] is None:
            top_axis = ax.secondary_xaxis("top")
            top_axis.set_xticks(group_centres)
            top_axis.set_xticklabels(group_labels)
            top_axis.tick_params(
                axis="x",
                length=0,
                pad=8,
                labelsize=11,
            )

            # Hide the secondary-axis border.
            top_axis.spines["top"].set_visible(False)

        ax.set_xlim(
            min(positions) - 0.7,
            max(positions) + 0.7,
        )

        fig.tight_layout()
        os.makedirs(out_dir, exist_ok=True)
        fig.savefig(
            os.path.join(out_dir, filename),
            dpi=150,
            bbox_inches="tight",
        )
        plt.close(fig)

        print(f"  Saved {filename}")

def plot_embedding_structure_boxplots(
    df_res: pd.DataFrame,
    out_dir: str,
) -> None:
    """
    Plot seed-level distributions of embedding-structure metrics.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import os

    method_order = [
        "km_greedy",
        "joe_km_greedy_sigmoid",
        "joe_km_greedy_linear",
        "joe_ohe_sigmoid",
        "joe_ohe_linear",
    ]

    method_labels = {
        "km_greedy": "KM",
        "joe_km_greedy_sigmoid": "JOECHAR_3SIG",
        "joe_km_greedy_linear": "JOECHAR_1LIN",
        "joe_ohe_sigmoid": "JOEOH_3SIG",
        "joe_ohe_linear": "JOEOH_1LIN",
    }

    plot_specs = [
        (
            "ari",
            "Adjusted Rand index",
            "ARI",
            "embedding_ari_boxplot.png",
            0.0,
            1.0,
        ),
        (
            "nmi",
            "Normalised mutual information",
            "NMI",
            "embedding_nmi_boxplot.png",
            0.0,
            1.0,
        ),
        (
            "silhouette_true",
            "True-cluster silhouette score",
            "Silhouette score",
            "embedding_silhouette_boxplot.png",
            -1.0,
            1.0,
        ),
        (
            "effect_distance_spearman",
            "Preservation of true effect distances",
            "Spearman correlation",
            "embedding_effect_distance_boxplot.png",
            -1.0,
            1.0,
        ),
    ]

    for (
        metric_prefix,
        title,
        ylabel,
        filename,
        lower_limit,
        upper_limit,
    ) in plot_specs:

        columns = [
            f"{metric_prefix}_{method}"
            for method in method_order
            if f"{metric_prefix}_{method}" in df_res.columns
        ]

        labels = [
            method_labels[method]
            for method in method_order
            if f"{metric_prefix}_{method}" in df_res.columns
        ]

        if not columns:
            print(
                f"  No columns found for {metric_prefix}; "
                f"skipping plot."
            )
            continue

        data = [
            df_res[column]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .to_numpy()
            for column in columns
        ]

        keep_indices = [
            i
            for i, values in enumerate(data)
            if len(values) > 0
        ]

        if not keep_indices:
            print(
                f"  No valid values found for {metric_prefix}; "
                f"skipping plot."
            )
            continue

        data = [data[i] for i in keep_indices]
        labels = [labels[i] for i in keep_indices]

        fig, ax = plt.subplots(
            figsize=(1.7 * len(data) + 3.0, 4.8)
        )

        boxplot = ax.boxplot(
            data,
            labels=labels,
            showmeans=False,
            patch_artist=True,
        )

        for box in boxplot["boxes"]:
            box.set(facecolor="lightgrey",
                    edgecolor="black",
                    alpha=1.0)

        for whisker in boxplot["whiskers"]:
            whisker.set(color="black")

        for cap in boxplot["caps"]:
            cap.set(color="black")

        for median in boxplot["medians"]:
            median.set(color="black", linewidth=2)

        for mean in boxplot["means"]:
            mean.set(marker="D",
                    markerfacecolor="black",
                    markeredgecolor="black",
                    markersize=5)

        ax.set_ylabel(ylabel, fontsize=11)
        ax.tick_params(axis="x", labelrotation=30)

        fig.tight_layout()
        os.makedirs(out_dir, exist_ok=True)
        fig.savefig(
            os.path.join(out_dir, filename),
            dpi=150,
            bbox_inches="tight",
        )
        plt.close(fig)

        print(f"  Saved {filename}")

In [5]:
df_res = combined_results_df

dataset_output_dir = os.path.join(
    RESULTS_ROOT,
    dataset_name,
)
out_dir = RESULTS_ROOT

# Call the plotting functions
plot_metric_gain_boxplots(df_res=df_res, out_dir=dataset_output_dir)
plot_embedding_structure_boxplots(df_res=df_res, out_dir=dataset_output_dir)

  Saved cindex_gain_boxplot.pdf
  Saved ibs_gain_boxplot.pdf
  No columns found for ari; skipping plot.
  No columns found for nmi; skipping plot.
  No columns found for silhouette_true; skipping plot.
  No columns found for effect_distance_spearman; skipping plot.


In [6]:
# Install r-base and rpy2 for R integration in Colab.
# This might be needed if R is not fully set up or R packages are missing.
!apt-get update
!apt-get install -y r-base
!pip install rpy2

# You may also need to install specific R packages required by your script.
# For example, if your R script uses 'survival' package, you would run:
# %load_ext rpy2.ipython
# %%R
# install.packages('survival')
# library(survival)

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cli.github.com/packages stable/main amd64 Packages [359 B]
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,005 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [72.8 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,566 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.2 MB]
Get:14 http://archive.ubunt

In [7]:
import shutil
import os

source_dir = RESULTS_ROOT
destination_dir = "/content/drive/MyDrive/plots_real_joe_km"

# Remove existing directory in Google Drive if it exists to avoid errors
if os.path.exists(destination_dir):
    shutil.rmtree(destination_dir)
    print(f"Removed existing directory: {destination_dir}")

# Copy the results directory to Google Drive
shutil.copytree(source_dir, destination_dir)
print(f"Plots saved to Google Drive at: {destination_dir}")

Plots saved to Google Drive at: /content/drive/MyDrive/plots_real_joe_km
